# Spotter — Freight Rate Prediction

Trains a freight-rate regression model on `data/train_test.csv`, generates point predictions for every load in `data/validation.csv`, and produces a 31-day rate forecast for the fixed December lane in `data/december_chart_inputs.csv`.

**Architecture at a glance:** two separately trained LightGBM models are used because the two target files don't share the same schema — `validation.csv` includes live market-condition columns (`market_index`, `quote_signal`) that don't exist for a 31-day-ahead forecast, since those signals aren't observable that far in advance.

- **Head A** (full feature set, incl. market signals) → used only for `validation.csv`
- **Head B** (market signals removed) → used only for the December forecast

Full write-up (methodology, validation approach, error analysis, and the December chart) is in `Spotter_Freight_Rate_Model_Report.pdf`.

## Setup

**Requirements:** Python 3.9+

**Dependencies:**
```bash
pip install pandas numpy scikit-learn lightgbm optuna holidays
```
(or `pip install -r requirements.txt` from the repo root — see the first code cell below, which installs the same set.)

**Input files** — place these in a `data/` subdirectory next to this notebook (provided by Spotter, not included in this repo):
- `data/train_test.csv`
- `data/validation.csv`
- `data/validation_predictions_template.csv`
- `data/december_chart_inputs.csv`

Spotter's `score.py` scorer is expected at the repo root, alongside this notebook.

**Run order:** top to bottom, one pass. There's no branching — every cell depends on state built by the cells above it (Head A must finish training before the "Predict on validation.csv" section, `FIT_STATS` must exist before any `engineer_features()` call, etc.).

**Outputs generated** (into `outputs/` and `models/`, created automatically if they don't exist):
- `outputs/validation_predictions.csv` — the 12,000 official predictions, `load_id,predicted_rate`
- `outputs/december_predictions.csv` — the 31-day December forecast
- `models/lgb_head_a.txt`, `models/lgb_head_b.txt` — serialized LightGBM models
- `outputs/validation_predictions_with_bands.csv`, `outputs/december_predictions_with_bands.csv` — P10/P90 prediction intervals, generated in the analysis section for the report only (not part of the official submission)

# Install packages

In [1]:
# Core dependencies for this notebook (see requirements.txt for pinned versions)
!pip install -q pandas numpy lightgbm holidays scikit-learn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 20.0 MB/s eta 0:00:00


# Load and take a first look

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/train_test.csv")
print(df.shape)
print(df.dtypes)
print(df.isna().sum())          # see missing values per column
df.head()

(48000, 14)
load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


In [3]:
print(df[df["weight"].isna()]["equipment"].value_counts())
print(df[df["market_index"].isna()]["date"].value_counts().head())
# missing weight/market_index rows aren't concentrated in one equipment type or date -> safe to impute rather than drop

equipment
Dry Van    175
Reefer      70
Flatbed     55
Name: count, dtype: int64
date
2025-06-15    5
2025-07-27    5
2025-04-15    4
2025-03-28    4
2025-10-02    4
Name: count, dtype: int64


# Data cleaning

`train_test.csv` (48,000 rows) has three data-quality issues, found via the `.isna().sum()` / value-count checks above:

1. **Missing `weight` (300 rows)** — imputed by **equipment-group median**, not a global median, because Dry Van, Reefer, and Flatbed carry very different typical loads (a global median would systematically bias lighter/heavier equipment types).
2. **Missing `market_index` (374 rows), scattered across ~300 different dates** — no single date or week is disproportionately affected, so this looks like sensor/reporting gaps rather than a structural issue. Imputed by global median.
3. **Negative `weight` (292 rows, found later in `sanity_check`)** — physically impossible, but recoverable: treated as missing and imputed the same way as case 1 (equipment-group median), rather than dropped.

**Rows are never dropped for missing `weight`/`market_index`/`quote_signal`/categoricals** — only for an unrecoverable `date` (can't build calendar features without one), or for `distance <= 0` / `posted_rate <= 0`, which can't be reconstructed from bad data (and `posted_rate` is the *target* — it must never be imputed, only dropped if invalid). Missing categoricals (`pickup`/`delivery`/`equipment`) get an explicit `"Unknown"` bucket instead of a drop, since LightGBM can split on that bucket meaningfully rather than losing the row entirely.

In [4]:
def clean_data(df):
    df = df.copy()

    # dates
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    # numeric columns — coerce bad values to NaN, then handle
    numeric_cols = ["distance", "weight", "pickup_lat", "pickup_lon",
                     "delivery_lat", "delivery_lon", "market_index", "quote_signal"]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Impute missing numerics with the column median rather than leaving NaN.
    # LightGBM *can* split on NaN natively, but explicit imputation makes the
    # behavior visible and explainable rather than left to the library's default
    # missing-value routing. Median (not mean) is used because distance/weight/
    # market_index are all right-skewed, so the mean would be pulled by outliers.
    for col in ["distance", "weight", "market_index", "quote_signal"]:
        df[col] = df[col].fillna(df[col].median())

    # Missing categoricals -> explicit "Unknown" bucket, never drop the row.
    # A dropped row loses a labeled training example for no good reason; an
    # "Unknown" category lets the model learn whatever signal is actually there
    # (e.g. "Unknown" pickups might correlate with a particular data source or era).
    for col in ["pickup", "delivery", "equipment"]:
        df[col] = df[col].fillna("Unknown")

    # drop rows only if date itself is unrecoverable (can't engineer time features without it)
    df = df.dropna(subset=["date"])

    return df

df_clean = clean_data(df)
print(df_clean.isna().sum())

load_id         0
pickup          0
delivery        0
pickup_lat      0
pickup_lon      0
delivery_lat    0
delivery_lon    0
distance        0
equipment       0
weight          0
date            0
market_index    0
quote_signal    0
posted_rate     0
dtype: int64


In [5]:
def sanity_check(df):
    df = df.copy()

    print("Before sanity checks:", df.shape)
    print("weight <= 0:", (df["weight"] <= 0).sum())
    print("distance <= 0:", (df["distance"] <= 0).sum())
    print("posted_rate <= 0:", (df["posted_rate"] <= 0).sum())

    # weight <= 0 -> physically impossible, but recoverable: impute by equipment-group median
    # (not a global median, since Reefer/Flatbed/Dry Van have very different typical loads)
    bad_weight = df["weight"] <= 0
    df.loc[bad_weight, "weight"] = np.nan
    df["weight"] = df.groupby("equipment")["weight"].transform(lambda x: x.fillna(x.median()))

    # distance <= 0 -> physically meaningless and there's no reliable way to
    # reconstruct a real route distance from the other columns, so drop.
    df = df[df["distance"] > 0]

    # posted_rate <= 0 -> this is the TARGET. Imputing a target value would mean
    # training the model to predict a number we made up -- always drop instead.
    df = df[df["posted_rate"] > 0]

    print("\nAfter sanity checks:", df.shape)
    return df

df_clean = sanity_check(df_clean)

Before sanity checks: (48000, 14)
weight <= 0: 292
distance <= 0: 0
posted_rate <= 0: 0

After sanity checks: (48000, 14)


# Feature engineering

Four groups of features are built, in two passes to stay leak-safe:

1. **Stateless / geometric** (`add_core_features`) — haversine distance, bearing, circuity ratio (actual distance vs. straight-line distance), distance/weight tiers, and calendar features (day-of-week, week-of-year, season, days-to-nearest-US-holiday). These are pure functions of a single row, so they're identical whether computed on train, holdout, or `validation.csv` — no fitting required, no leakage possible.
2. **Fitted / historical-stats** (`fit_feature_stats` + `apply_fitted_features`) — lane-level, origin-level, and destination-level historical average rates, shrunk toward the global mean by frequency (see the `k=10` comment below), plus a coarse geographic region grid and "hub-ness" counts. These **must** be fit only on `df_train` and then mapped onto everything else, or the holdout/validation metrics would be inflated by information the model shouldn't have access to at prediction time.
3. **Interaction terms** (`equip_x_disttier`, `equip_x_season`, `market_x_distance`) — combinations that let the tree split on equipment-specific seasonal or distance-tier effects in a single node instead of learning them from scratch across multiple splits.
4. **Gated features** (`quote_x_equipment`, added later) — held back until the `quote_signal` circularity check (below) confirmed it was safe to use.

`engineer_features()` is the single entry point that wraps steps 1–2 and is called identically on train, holdout, `validation.csv`, and the December file.

In [6]:
import numpy as np
import pandas as pd
import holidays

# ---------------------------------------------------------------
# 0. Time-based split: train on everything before December,
#    holdout = December (mirrors the real validation.csv scenario)
# ---------------------------------------------------------------
holdout_start = pd.Timestamp(year=2025, month=10, day=1)
df_train   = df_clean[df_clean["date"] <  holdout_start].copy()
df_holdout = df_clean[df_clean["date"] >= holdout_start].copy()
print("train:", df_train.shape, "| holdout (Oct):", df_holdout.shape)

# ---------------------------------------------------------------
# 1. STATELESS core features — safe to compute identically on
#    train / holdout / validation.csv, no fitting required
# ---------------------------------------------------------------
R_EARTH_MI = 3958.8

def haversine_miles(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R_EARTH_MI * 2 * np.arcsin(np.sqrt(a))

def bearing_deg(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1)*np.sin(lat2) - np.sin(lat1)*np.cos(lat2)*np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

def days_to_nearest_holiday(dates):
    years = dates.dt.year.unique()
    yr_range = sorted(set(years) | {y-1 for y in years} | {y+1 for y in years})
    us_hol = holidays.US(years=yr_range)
    hol_arr = pd.to_datetime(sorted(us_hol.keys())).values.astype("datetime64[D]")
    date_arr = dates.values.astype("datetime64[D]")
    diffs = np.abs(date_arr[:, None] - hol_arr[None, :]).astype("timedelta64[D]").astype(int)
    return diffs.min(axis=1)

def add_core_features(df):
    df = df.copy()

    # --- geo / route ---
    df["haversine_mi"]   = haversine_miles(df.pickup_lat, df.pickup_lon, df.delivery_lat, df.delivery_lon)
    df["circuity_ratio"] = df["distance"] / df["haversine_mi"].replace(0, np.nan)
    df["circuity_ratio"] = df["circuity_ratio"].fillna(1.0)
    df["bearing_deg"]    = bearing_deg(df.pickup_lat, df.pickup_lon, df.delivery_lat, df.delivery_lon)

    # --- distance tier (fixed business cutoffs — interpretable, stable across splits) ---
    df["distance_tier"] = pd.cut(df["distance"], bins=[0, 250, 450, np.inf],
                                  labels=["short", "mid", "long"])

    # --- weight ---
    df["weight_to_distance"] = df["weight"] / df["distance"]
    df["weight_class"] = pd.cut(df["weight"], bins=[0, 10000, 45000, np.inf],
                                 labels=["light", "standard", "overweight"])

    # --- calendar ---
    df["day_of_week"]    = df["date"].dt.dayofweek          # 0=Mon
    df["month"]          = df["date"].dt.month
    df["week_of_year"]   = df["date"].dt.isocalendar().week.astype(int)
    df["is_month_end"]   = (df["date"].dt.day >= df["date"].dt.days_in_month - 2).astype(int)
    df["days_to_holiday"] = days_to_nearest_holiday(df["date"])
    df["season"] = df["month"].map({12:"winter",1:"winter",2:"winter",
                                     3:"spring",4:"spring",5:"spring",
                                     6:"summer",7:"summer",8:"summer",
                                     9:"fall",10:"fall",11:"fall"})

    # --- interactions ---
    df["equip_x_disttier"] = df["equipment"].astype(str) + "_" + df["distance_tier"].astype(str)
    df["equip_x_season"]   = df["equipment"].astype(str) + "_" + df["season"].astype(str)
    df["market_x_distance"] = df["market_index"] * df["distance"]

    return df

train: (43147, 14) | holdout (Oct): (4853, 14)


In [7]:
# ---------------------------------------------------------------
# 2. FIT step — region grid + leak-safe historical stats,
#    fit ONLY on df_train, then mapped onto everything else
# ---------------------------------------------------------------
def fit_feature_stats(df_train, k=10):
    stats = {}

    # -- self-contained region grid: tercile bins on train pickup+delivery coords --
    lats = pd.concat([df_train["pickup_lat"], df_train["delivery_lat"]])
    lons = pd.concat([df_train["pickup_lon"], df_train["delivery_lon"]])
    lat_edges = np.quantile(lats, [0, 1/3, 2/3, 1]); lat_edges[0]-=1; lat_edges[-1]+=1
    lon_edges = np.quantile(lons, [0, 1/3, 2/3, 1]); lon_edges[0]-=1; lon_edges[-1]+=1
    stats["lat_edges"], stats["lon_edges"] = lat_edges, lon_edges

    global_mean = df_train["posted_rate"].mean()
    stats["global_mean"] = global_mean

    # -- lane (O-D) stats, shrunk toward global mean by frequency --
    lane = df_train.groupby(["pickup", "delivery"])["posted_rate"].agg(["mean", "std", "count"])
    lane["lane_mean_shrunk"] = (lane["mean"]*lane["count"] + global_mean*k) / (lane["count"] + k)
    lane["lane_std"] = lane["std"].fillna(0)
    stats["lane_stats"] = lane.rename(columns={"count": "lane_freq"})[["lane_mean_shrunk","lane_std","lane_freq"]]

    # -- origin / destination shrunk means --
    o = df_train.groupby("pickup")["posted_rate"].agg(["mean", "count"])
    o["origin_mean_shrunk"] = (o["mean"]*o["count"] + global_mean*k) / (o["count"] + k)
    stats["origin_stats"] = o[["origin_mean_shrunk"]]

    d = df_train.groupby("delivery")["posted_rate"].agg(["mean", "count"])
    d["dest_mean_shrunk"] = (d["mean"]*d["count"] + global_mean*k) / (d["count"] + k)
    stats["dest_stats"] = d[["dest_mean_shrunk"]]

    # -- hub-ness: raw appearance count as pickup OR delivery --
    hub = df_train["pickup"].value_counts().add(df_train["delivery"].value_counts(), fill_value=0)
    stats["hub_counts"] = hub

    return stats

def region_label(lat, lon, lat_edges, lon_edges):
    lat_bin = pd.cut(lat, bins=lat_edges, labels=["S", "Mid", "N"])
    lon_bin = pd.cut(lon, bins=lon_edges, labels=["W", "Cent", "E"])
    return lat_bin.astype(str) + "-" + lon_bin.astype(str)

def apply_fitted_features(df, stats):
    df = df.copy()

    # region + regional pair
    df["pickup_region"]   = region_label(df["pickup_lat"], df["pickup_lon"], stats["lat_edges"], stats["lon_edges"])
    df["delivery_region"] = region_label(df["delivery_lat"], df["delivery_lon"], stats["lat_edges"], stats["lon_edges"])
    df["regional_pair"]   = df["pickup_region"] + "->" + df["delivery_region"]

    # lane stats (merge, fallback to global mean / 0 for unseen lanes)
    df = df.merge(stats["lane_stats"], left_on=["pickup","delivery"], right_index=True, how="left")
    df["lane_mean_shrunk"] = df["lane_mean_shrunk"].fillna(stats["global_mean"])
    df["lane_std"]         = df["lane_std"].fillna(0)
    df["lane_freq"]        = df["lane_freq"].fillna(0)

    # origin / dest stats
    df = df.merge(stats["origin_stats"], left_on="pickup", right_index=True, how="left")
    df["origin_mean_shrunk"] = df["origin_mean_shrunk"].fillna(stats["global_mean"])

    df = df.merge(stats["dest_stats"], left_on="delivery", right_index=True, how="left")
    df["dest_mean_shrunk"] = df["dest_mean_shrunk"].fillna(stats["global_mean"])

    # hub-ness (unseen city -> 0)
    df["hub_pickup"]   = df["pickup"].map(stats["hub_counts"]).fillna(0)
    df["hub_delivery"] = df["delivery"].map(stats["hub_counts"]).fillna(0)

    return df

def engineer_features(df, stats):
    df = add_core_features(df)
    df = apply_fitted_features(df, stats)
    return df


In [8]:
# Sanity check: confirm the time-based split boundary lands where expected
print("clean date range:", df_clean["date"].min(), "->", df_clean["date"].max())
print("df_clean year max:", df_clean["date"].dt.year.max())
print("holdout_start:", holdout_start)
print("df_train shape:", df_train.shape)
print("df_holdout shape:", df_holdout.shape)

clean date range: 2025-01-01 00:00:00 -> 2025-10-31 00:00:00
df_clean year max: 2025
holdout_start: 2025-10-01 00:00:00
df_train shape: (43147, 14)
df_holdout shape: (4853, 14)


In [9]:
# ---------------------------------------------------------------
# 3. Fit on train ONLY, transform train + holdout
# ---------------------------------------------------------------
FIT_STATS = fit_feature_stats(df_train)
df_train_fe   = engineer_features(df_train, FIT_STATS)
df_holdout_fe = engineer_features(df_holdout, FIT_STATS)

print(df_train_fe.shape, df_holdout_fe.shape)
df_train_fe.head()

(43147, 39) (4853, 39)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,...,pickup_region,delivery_region,regional_pair,lane_mean_shrunk,lane_std,lane_freq,origin_mean_shrunk,dest_mean_shrunk,hub_pickup,hub_delivery
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,...,Mid-E,N-E,Mid-E->N-E,1446.979741,598.881432,15,2297.676424,2653.714277,2013,856
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,...,Mid-E,N-E,Mid-E->N-E,1110.727987,46.087095,28,2297.676424,2541.221037,2013,1711
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,...,N-E,N-Cent,N-E->N-Cent,2209.539084,164.971902,8,2509.499628,2171.606855,1711,1194
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,...,N-E,Mid-Cent,N-E->Mid-Cent,2142.014485,168.387393,22,2687.438812,1812.970616,2068,1894
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,...,S-W,Mid-Cent,S-W->Mid-Cent,1908.188529,62.111310,8,2153.800584,1807.882833,530,1982


In [11]:
# Confirm validation.csv sits entirely after the training window (Nov-Dec 2025) -
# this is what the Oct-2025 holdout above is meant to approximate
val = pd.read_csv("data/validation.csv")
val["date"] = pd.to_datetime(val["date"], errors="coerce")
print(val["date"].min(), "->", val["date"].max())

2025-11-01 00:00:00 -> 2025-12-31 00:00:00


In [12]:
# quote_signal circularity check: if this were another system's internal price
# estimate we'd expect a correlation near +/-1.0. r ~= -0.11 is a weak relationship,
# so it's kept as a Head A feature (see report Section 2.3)
print(df_train["quote_signal"].corr(df_train["posted_rate"]))

-0.1094588863046443


In [13]:
def add_gated_features(df):
    df = df.copy()
    # "Gated" because this interaction is only added after the correlation check
    # above cleared quote_signal for use -- if that check had shown near-1.0
    # correlation (suggesting quote_signal IS another system's price estimate),
    # this feature would leak the answer and should never be built at all.
    df["quote_x_equipment"] = df["quote_signal"] * df["equipment"].astype("category").cat.codes
    return df

df_train_fe   = add_gated_features(df_train_fe)
df_holdout_fe = add_gated_features(df_holdout_fe)

# Head A — LightGBM (full feature set)

Trained with every engineered feature, including `market_index`, `quote_signal`, and their interaction terms. This head generates the 12,000-row `validation_predictions.csv`.

## Model Selection

Before committing to LightGBM, three gradient-boosted tree families were evaluated head-to-head on the identical October 2025 holdout:

| Model | Holdout MAPE | Verdict |
|---|---|---|
| **LightGBM** | **5.20%** | **Selected** — lowest error, fastest training/iteration |
| CatBoost | 7.74% | Rejected — underperformed on this feature set |
| XGBoost | 9.02% | Rejected — underperformed both alternatives |
| CatBoost + LightGBM ensemble | 6.02% | Rejected — worse than LightGBM alone (CatBoost's weaker standalone predictions dragged the blend down) |

CatBoost was actually the initial front-runner going in, since its ordered target statistics handle the high-cardinality `pickup`/`delivery` columns without the leakage risk of naive target encoding — so it wasn't dismissed casually, it was tested and lost on holdout evidence. The CatBoost/XGBoost/ensemble training code lived in a separate exploratory notebook (not included here for brevity) — the table above and the full baseline ablation against non-ML/linear baselines are reproduced in Section 4 of the report.

**Why LightGBM specifically, beyond the raw metric:**
- Native categorical feature support (`categorical_feature=cat_features`) — no one-hot encoding needed for high-cardinality columns like `pickup`/`delivery` (64 unique cities each), which would otherwise blow up the feature space and dilute per-category sample size.
- Histogram-based splitting makes it meaningfully faster to iterate on than CatBoost at this dataset size (~43K training rows), which mattered for the hyperparameter search below.
- Handles the mix of numeric, categorical, and NaN-containing columns without a separate preprocessing pipeline.

In [14]:
drop_cols = ["load_id", "posted_rate", "date", "rate_per_mile"]
target_col = "posted_rate"
feature_cols = [c for c in df_train_fe.columns if c not in drop_cols]

# Columns passed to LightGBM as native categoricals rather than left as strings/
# numerics. Two groups: (a) genuinely categorical identity columns (pickup,
# delivery, equipment, the engineered region/tier labels) where there's no
# meaningful ordering, and (b) the interaction columns built by concatenating
# strings (equip_x_disttier, equip_x_season), which are categorical by
# construction. Numeric features (distance, weight, market_index, the
# shrunk lane/origin/dest means, etc.) are deliberately left OUT of this list
# so LightGBM treats them as ordered/continuous, which they genuinely are.
cat_features = [
    "pickup", "delivery", "equipment",
    "distance_tier", "weight_class", "season",
    "pickup_region", "delivery_region", "regional_pair",
    "equip_x_disttier", "equip_x_season"
]
cat_features = [c for c in cat_features if c in feature_cols]

X_train, y_train = df_train_fe[feature_cols], df_train_fe[target_col]
X_holdout, y_holdout = df_holdout_fe[feature_cols], df_holdout_fe[target_col]

print(X_train.shape, X_holdout.shape)
print("cat_features:", cat_features)

(43147, 37) (4853, 37)
cat_features: ['pickup', 'delivery', 'equipment', 'distance_tier', 'weight_class', 'season', 'pickup_region', 'delivery_region', 'regional_pair', 'equip_x_disttier', 'equip_x_season']


In [15]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

# LightGBM needs categorical columns as pandas 'category' dtype, not raw strings
X_train_lgb = X_train.copy()
X_holdout_lgb = X_holdout.copy()
for col in cat_features:
    X_train_lgb[col] = X_train_lgb[col].astype("category")
    X_holdout_lgb[col] = X_holdout_lgb[col].astype("category")

lgb_train = lgb.Dataset(X_train_lgb, y_train, categorical_feature=cat_features)
lgb_val   = lgb.Dataset(X_holdout_lgb, y_holdout, categorical_feature=cat_features, reference=lgb_train)

# Manually-tuned starting point (later checked against an Optuna search in the
# analysis section below, which found a ~0.3pp improvement over these values).
lgb_params = {
    "objective": "mape",        # optimizes MAPE directly rather than RMSE, since MAPE is
                                # the headline evaluation metric and freight rates span a
                                # wide dollar range where a fixed-dollar error matters less
                                # on a $10k load than on a $200 one
    "metric": "mape",           # what early stopping watches on the holdout set
    "learning_rate": 0.05,      # conservative step size; early stopping (below) finds the
                                # right number of boosting rounds rather than relying on a
                                # fast learning rate to converge in fewer rounds
    "num_leaves": 64,           # controls tree complexity / capacity per boosting round
    "max_depth": 8,             # caps how deep any single tree can go, as a guard against
                                # overfitting on top of num_leaves
    "min_data_in_leaf": 30,     # minimum samples per leaf; prevents the model from carving
                                # out leaves that memorize a handful of rows
    "feature_fraction": 0.85,   # randomly samples 85% of features per tree (like Random
                                # Forest's feature bagging) -- adds regularization and
                                # decorrelates trees in the ensemble
    "bagging_fraction": 0.85,   # randomly samples 85% of rows per iteration, same rationale
    "bagging_freq": 5,          # how often (every N iterations) bagging is re-applied
    "seed": 42,                 # reproducibility
    "verbose": -1,
}

lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=8000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "holdout"],
    callbacks=[lgb.early_stopping(stopping_rounds=150), lgb.log_evaluation(200)],
)

preds_lgb = lgb_model.predict(X_holdout_lgb, num_iteration=lgb_model.best_iteration)

rmse_lgb = mean_squared_error(y_holdout, preds_lgb) ** 0.5
mae_lgb  = mean_absolute_error(y_holdout, preds_lgb)
mape_lgb = mean_absolute_percentage_error(y_holdout, preds_lgb) * 100

print(f"LightGBM Holdout RMSE: {rmse_lgb:.2f}")
print(f"LightGBM Holdout MAE:  {mae_lgb:.2f}")
print(f"LightGBM Holdout MAPE: {mape_lgb:.2f}%")

Training until validation scores don't improve for 150 rounds
[200]	train's mape: 0.0452899	holdout's mape: 0.0558324
[400]	train's mape: 0.0418292	holdout's mape: 0.0537173
[600]	train's mape: 0.0406722	holdout's mape: 0.0530283
[800]	train's mape: 0.0401496	holdout's mape: 0.052762
[1000]	train's mape: 0.0397341	holdout's mape: 0.0525156
[1200]	train's mape: 0.0394585	holdout's mape: 0.0524043
[1400]	train's mape: 0.0392737	holdout's mape: 0.0522915
[1600]	train's mape: 0.0390872	holdout's mape: 0.0521779
[1800]	train's mape: 0.0388995	holdout's mape: 0.0520211
[2000]	train's mape: 0.0387528	holdout's mape: 0.0519612
Early stopping, best iteration is:
[2027]	train's mape: 0.0387409	holdout's mape: 0.0519547
LightGBM Holdout RMSE: 653.25
LightGBM Holdout MAE:  120.61
LightGBM Holdout MAPE: 5.20%


In [16]:
lgb_pct_error = np.abs(y_holdout - preds_lgb) / y_holdout * 100
print(lgb_pct_error.describe(percentiles=[.5, .75, .90, .95, .99]))

count    4853.000000
mean        5.195471
std        28.730461
min         0.000181
50%         1.748992
75%         3.121076
90%         4.923164
95%         6.861892
99%        77.506213
max       460.708279
Name: posted_rate, dtype: float64


In [17]:
error_df_a = df_holdout_fe[[
    "load_id", "pickup", "delivery", "distance",
    "equipment", "posted_rate"
]].copy()

error_df_a["predicted_rate"] = preds_lgb

error_df_a["abs_error"] = (
    error_df_a["posted_rate"] - error_df_a["predicted_rate"]
).abs()

error_df_a["pct_error"] = (
    error_df_a["abs_error"] / error_df_a["posted_rate"] * 100
)


print("=== Head A — overall error distribution ===")
print(
    error_df_a["pct_error"].describe(
        percentiles=[.5, .75, .90, .95, .99]
    )
)


print("\n=== Head A — how many loads fall in each error band ===")

bands_a = pd.cut(
    error_df_a["pct_error"],
    bins=[0, 5, 10, 20, 50, 100, np.inf],
    labels=[
        "0-5%",
        "5-10%",
        "10-20%",
        "20-50%",
        "50-100%",
        ">100%"
    ]
)

print(bands_a.value_counts().sort_index())

print(
    (
        bands_a.value_counts().sort_index()
        / len(error_df_a) * 100
    ).round(2).astype(str) + "%"
)


print("\n=== Head A — worst 15 predictions (by absolute $ error) ===")

print(
    error_df_a
    .sort_values("abs_error", ascending=False)
    .head(15)[[
        "pickup",
        "delivery",
        "distance",
        "equipment",
        "posted_rate",
        "predicted_rate",
        "abs_error",
        "pct_error"
    ]]
)

=== Head A — overall error distribution ===
count    4853.000000
mean        5.195471
std        28.730461
min         0.000181
50%         1.748992
75%         3.121076
90%         4.923164
95%         6.861892
99%        77.506213
max       460.708279
Name: pct_error, dtype: float64

=== Head A — how many loads fall in each error band ===
pct_error
0-5%       4383
5-10%       359
10-20%       31
20-50%        0
50-100%      39
>100%        41
Name: count, dtype: int64
pct_error
0-5%       90.32%
5-10%        7.4%
10-20%      0.64%
20-50%       0.0%
50-100%      0.8%
>100%       0.84%
Name: count, dtype: object

=== Head A — worst 15 predictions (by absolute $ error) ===
               pickup     delivery  distance equipment  posted_rate  \
43539       Milwaukee  Bakersfield    1893.0   Dry Van     17514.11   
46213        Columbia       Tucson    2023.4   Flatbed     17893.17   
47298          Boston  Bakersfield    2979.1    Reefer     19110.30   
46516          Toledo  Albuquerque 

In [ ]:
importances_lgb = pd.DataFrame({
    "feature": lgb_model.feature_name(),
    "importance": lgb_model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False)
print(importances_lgb.head(20))

importances_lgb["importance_pct"] = importances_lgb["importance"] / importances_lgb["importance"].sum() * 100
print(importances_lgb[["feature", "importance_pct"]].head(20))

In [ ]:
# Lane-leakage check: the Lexington -> Fort Wayne lane (used for the December
# forecast) has enough historical support in train_test.csv that a shrinkage-
# adjusted lane average is a legitimate feature rather than noise (report Section 2.2)
print(df_clean[(df_clean["pickup"]=="Lexington") & (df_clean["delivery"]=="Fort Wayne")].shape[0])

# Predict on validation.csv (Head A)

In [ ]:
val_raw = pd.read_csv("data/validation.csv")
print(val_raw.shape)
print(val_raw.isna().sum())

val_clean = clean_data(val_raw)   # same function used on train_test.csv
# NOTE: validation.csv has no posted_rate, so sanity_check() must be adapted —
# skip the posted_rate <= 0 drop, since that column doesn't exist here
print(val_clean.isna().sum())
print(val_clean.shape)

In [ ]:
val_fe = engineer_features(val_clean, FIT_STATS)   # FIT_STATS from df_train, already fitted
val_fe = add_gated_features(val_fe)                # adds quote_x_equipment etc.

print(val_fe.shape)
val_fe.head()  # quick visual check only, not required

# Generate the official validation_predictions.csv

In [ ]:
X_val = val_fe[feature_cols].copy()
for col in cat_features:
    X_val[col] = X_val[col].astype("category")

val_predictions = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)

print(pd.Series(val_predictions).describe())   # sanity check: no negatives, reasonable range

count    12000.000000
mean      2313.130355
std       1307.107989
min        198.755903
25%       1260.212699
50%       2019.111593
75%       3285.350333
max       5982.791074
dtype: float64


In [ ]:
template = pd.read_csv("data/validation_predictions_template.csv")
print(template.shape)   # should be (12000, 2)

val_fe["predicted_rate"] = val_predictions
result = template[["load_id"]].merge(val_fe[["load_id", "predicted_rate"]], on="load_id", how="left")

print(result.isna().sum())                     # must be zero
print(result.shape)                             # must be exactly (12000, 2)
print((result["predicted_rate"] <= 0).sum())    # must be zero
print(list(result.columns))                     # must be exactly ['load_id', 'predicted_rate']

import os
os.makedirs("outputs", exist_ok=True)
result.to_csv("outputs/validation_predictions.csv", index=False)

In [ ]:
# Save Head A so it can be reused without retraining
import os
os.makedirs("models", exist_ok=True)
lgb_model.save_model("models/lgb_head_a.txt")

*(`score.py` is Spotter's own scorer, expected to already be present in the working directory — see the repo README for where to get it.)*

# Head B — LightGBM (no market signals)

## Why two models, not one?

`validation.csv` includes `market_index` and `quote_signal`. `december_chart_inputs.csv` does **not** — those signals aren't observable 31 days into the future, so they simply don't exist for that file. Three ways to handle this were considered:

1. **Impute the missing columns for December** (e.g. carry forward the last known market_index) — rejected, because it would inject a fabricated number the model would treat as real signal, quietly biasing the December forecast toward whatever value was imputed.
2. **Train one model to be robust to missing columns** (e.g. mask them out at random during training) — rejected, because it spends model capacity learning to tolerate missingness instead of learning the actual rate relationships, and doesn't mirror how the two files actually differ (December isn't "market_index randomly missing," it's "market_index never available for this horizon").
3. **Train two separate models, one per inference scenario** — chosen. This mirrors the real production constraint directly: a cross-sectional pricing call (validation.csv-style) has live market data available; a 31-day-ahead forecast never will.

**Head B** is trained on identical rows and the identical target, but with `market_index`, `quote_signal`, and their derived interaction terms (`market_x_distance`, `quote_x_equipment`) removed entirely from the feature set. It's used exclusively for the December forecast. Both heads share the same cleaning pipeline, feature engineering, and time-based train/holdout split — they differ only in which columns are available to split on.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error


In [ ]:
market_derived = ["market_index", "quote_signal", "market_x_distance", "quote_x_equipment"]

feature_cols_b = [c for c in feature_cols if c not in market_derived]
cat_features_b = [c for c in cat_features if c in feature_cols_b]

print("Dropped:", [c for c in feature_cols if c in market_derived])
print("Head B feature count:", len(feature_cols_b), "(was", len(feature_cols), ")")

Dropped: ['market_index', 'quote_signal', 'market_x_distance', 'quote_x_equipment']
Head B feature count: 33 (was 37 )


In [ ]:
X_train_b = X_train_lgb[feature_cols_b].copy()
X_holdout_b = X_holdout_lgb[feature_cols_b].copy()

print("X_train_b shape:", X_train_b.shape)
print("X_holdout_b shape:", X_holdout_b.shape)

X_train_b shape: (43147, 33)
X_holdout_b shape: (4853, 33)


In [ ]:
lgb_train_b = lgb.Dataset(X_train_b, y_train, categorical_feature=cat_features_b)
lgb_val_b   = lgb.Dataset(X_holdout_b, y_holdout, categorical_feature=cat_features_b, reference=lgb_train_b)

lgb_params_b = {
    "objective": "mape",
    "metric": "mape",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "max_depth": 8,
    "min_data_in_leaf": 30,
    "feature_fraction": 0.85,
    "bagging_fraction": 0.85,
    "bagging_freq": 5,
    "seed": 42,
    "verbose": -1,
}

lgb_model_b = lgb.train(
    lgb_params_b,
    lgb_train_b,
    num_boost_round=8000,
    valid_sets=[lgb_train_b, lgb_val_b],
    valid_names=["train", "holdout"],
    callbacks=[lgb.early_stopping(stopping_rounds=150), lgb.log_evaluation(200)],
)

Training until validation scores don't improve for 150 rounds
[200]	train's mape: 0.0488853	holdout's mape: 0.060035
[400]	train's mape: 0.043554	holdout's mape: 0.0563874
[600]	train's mape: 0.042104	holdout's mape: 0.0558835
[800]	train's mape: 0.0412293	holdout's mape: 0.055317
[1000]	train's mape: 0.0407302	holdout's mape: 0.0550702
Early stopping, best iteration is:
[1025]	train's mape: 0.0406824	holdout's mape: 0.0550385


In [ ]:
preds_lgb_b = lgb_model_b.predict(X_holdout_b, num_iteration=lgb_model_b.best_iteration)

rmse_b = mean_squared_error(y_holdout, preds_lgb_b) ** 0.5
mae_b  = mean_absolute_error(y_holdout, preds_lgb_b)
mape_b = mean_absolute_percentage_error(y_holdout, preds_lgb_b) * 100

print(f"Head B Holdout RMSE: {rmse_b:.2f}")
print(f"Head B Holdout MAE:  {mae_b:.2f}")
print(f"Head B Holdout MAPE: {mape_b:.2f}%")

pct_error_b = np.abs(y_holdout - preds_lgb_b) / y_holdout * 100
print(pct_error_b.describe(percentiles=[.5, .75, .90, .95, .99]))

Head B Holdout RMSE: 653.83
Head B Holdout MAE:  124.19
Head B Holdout MAPE: 5.50%
count    4853.000000
mean        5.503851
std        28.923212
min         0.001116
50%         1.981221
75%         3.610972
90%         5.741628
95%         7.378619
99%        77.315488
max       486.076528
Name: posted_rate, dtype: float64


In [ ]:
error_df_b = df_holdout_fe[["load_id","pickup","delivery","distance","equipment","posted_rate"]].copy()
error_df_b["predicted_rate"] = preds_lgb_b
error_df_b["abs_error"]      = (error_df_b["posted_rate"] - error_df_b["predicted_rate"]).abs()
error_df_b["pct_error"]      = error_df_b["abs_error"] / error_df_b["posted_rate"] * 100

print("=== Head B — overall error distribution ===")
print(error_df_b["pct_error"].describe(percentiles=[.5, .75, .90, .95, .99]))

print("\n=== Head B — how many loads fall in each error band ===")
bands_b = pd.cut(error_df_b["pct_error"], bins=[0,5,10,20,50,100,np.inf],
                  labels=["0-5%","5-10%","10-20%","20-50%","50-100%",">100%"])
print(bands_b.value_counts().sort_index())
print((bands_b.value_counts().sort_index() / len(error_df_b) * 100).round(2).astype(str) + "%")

print("\n=== Head B — worst 15 predictions (by absolute $ error) ===")
print(error_df_b.sort_values("abs_error", ascending=False).head(15)
      [["pickup","delivery","distance","equipment","posted_rate","predicted_rate","abs_error","pct_error"]])

=== Head B — overall error distribution ===
count    4853.000000
mean        5.503851
std        28.923212
min         0.001116
50%         1.981221
75%         3.610972
90%         5.741628
95%         7.378619
99%        77.315488
max       486.076528
Name: pct_error, dtype: float64

=== Head B — how many loads fall in each error band ===
pct_error
0-5%       4193
5-10%       541
10-20%       39
20-50%        0
50-100%      39
>100%        41
Name: count, dtype: int64
pct_error
0-5%        86.4%
5-10%      11.15%
10-20%       0.8%
20-50%       0.0%
50-100%      0.8%
>100%       0.84%
Name: count, dtype: object

=== Head B — worst 15 predictions (by absolute $ error) ===
               pickup     delivery  distance equipment  posted_rate  \
43539       Milwaukee  Bakersfield    1893.0   Dry Van     17514.11   
46213        Columbia       Tucson    2023.4   Flatbed     17893.17   
47298          Boston  Bakersfield    2979.1    Reefer     19110.30   
46516          Toledo  Albuquerque 

In [ ]:
importances_lgb_b = pd.DataFrame({
    "feature": lgb_model_b.feature_name(),
    "importance": lgb_model_b.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False)

importances_lgb_b["importance_pct"] = importances_lgb_b["importance"] / importances_lgb_b["importance"].sum() * 100
print(importances_lgb_b[["feature", "importance_pct"]].head(20))

               feature  importance_pct
6             distance       38.718373
9         haversine_mi       13.438181
8               weight        5.403575
17        week_of_year        3.700021
10      circuity_ratio        3.630623
0               pickup        3.471253
1             delivery        3.446853
19     days_to_holiday        2.974370
13  weight_to_distance        2.826822
12       distance_tier        2.767731
22      equip_x_season        2.486289
21    equip_x_disttier        1.802941
15         day_of_week        1.683935
27            lane_std        1.583385
26    lane_mean_shrunk        1.413460
25       regional_pair        1.357860
7            equipment        1.113948
11         bearing_deg        1.057793
16               month        0.999356
32        hub_delivery        0.818702


In [ ]:
comparison = pd.DataFrame({
    "Head A (full features)": {
        "MAPE": "5.20%", "RMSE": 653.25, "MAE": 120.61,
        "median pct_error": "1.75%", "95th pct_error": "6.86%", "99th pct_error": "77.51%"
    },
    "Head B (no market signals)": {
        "MAPE": "5.50%", "RMSE": 653.83, "MAE": 124.19,
        "median pct_error": "1.98%", "95th pct_error": "7.38%", "99th pct_error": "77.32%"
    }
})
print(comparison)

                 Head A (full features) Head B (no market signals)
MAPE                              5.20%                      5.50%
RMSE                             653.25                     653.83
MAE                              120.61                     124.19
median pct_error                  1.75%                      1.98%
95th pct_error                    6.86%                      7.38%
99th pct_error                   77.51%                     77.32%


In [ ]:
lgb_model_b.save_model("models/lgb_head_b.txt")

In [ ]:
dec_raw = pd.read_csv("data/december_chart_inputs.csv")
print(dec_raw.shape)
print(dec_raw.columns.tolist())

dec_clean = dec_raw.copy()
dec_clean["date"] = pd.to_datetime(dec_clean["date"], errors="coerce")
for col in ["distance", "weight"]:
    dec_clean[col] = pd.to_numeric(dec_clean[col], errors="coerce")
for col in ["pickup", "delivery", "equipment"]:
    dec_clean[col] = dec_clean[col].fillna("Unknown")

# ---------------------------------------------------------------
# december_chart_inputs.csv has NO lat/lon columns, but the feature
# pipeline needs them. Lexington/Fort Wayne are real cities that
# already exist in train_test.csv, so look their coords up from there
# instead of guessing — this stays fully leak-safe since we're only
# taking the fixed GPS location of a named city, not any rate info.
# ---------------------------------------------------------------
city_coords = pd.concat([
    df_clean[["pickup", "pickup_lat", "pickup_lon"]].rename(
        columns={"pickup": "city", "pickup_lat": "lat", "pickup_lon": "lon"}),
    df_clean[["delivery", "delivery_lat", "delivery_lon"]].rename(
        columns={"delivery": "city", "delivery_lat": "lat", "delivery_lon": "lon"}),
]).drop_duplicates(subset="city")

# sanity check: each city name should map to ONE coordinate, not several
dupe_check = pd.concat([
    df_clean[["pickup", "pickup_lat", "pickup_lon"]].rename(
        columns={"pickup": "city", "pickup_lat": "lat", "pickup_lon": "lon"}),
    df_clean[["delivery", "delivery_lat", "delivery_lon"]].rename(
        columns={"delivery": "city", "delivery_lat": "lat", "delivery_lon": "lon"}),
]).groupby("city")[["lat", "lon"]].nunique()
inconsistent = dupe_check[(dupe_check["lat"] > 1) | (dupe_check["lon"] > 1)]
if len(inconsistent) > 0:
    print("WARNING: cities with inconsistent coordinates in training data:")
    print(inconsistent)
else:
    print("OK: every city maps to exactly one lat/lon in train_test.csv")

lookup = city_coords.set_index("city")[["lat", "lon"]]

for city_col, lat_col, lon_col in [("pickup", "pickup_lat", "pickup_lon"),
                                     ("delivery", "delivery_lat", "delivery_lon")]:
    missing = ~dec_clean[city_col].isin(lookup.index)
    if missing.any():
        raise ValueError(f"Cities in December file not found in training data: {dec_clean.loc[missing, city_col].unique()}")
    dec_clean[lat_col] = dec_clean[city_col].map(lookup["lat"])
    dec_clean[lon_col] = dec_clean[city_col].map(lookup["lon"])

# ---------------------------------------------------------------
# add_core_features() unconditionally computes market_x_distance =
# market_index * distance. December has no market_index (by design —
# that's the whole reason Head B exists), so add a placeholder purely
# so the function doesn't crash. The value itself is irrelevant:
# market_index and market_x_distance are both dropped from
# feature_cols_b before prediction, so Head B never sees this number.
# ---------------------------------------------------------------
dec_clean["market_index"] = np.nan

print(dec_clean.isna().sum())
print(dec_clean.shape)  # should be (31, 9) now — 6 original + pickup/delivery lat/lon + market_index placeholder
dec_clean.head()

In [ ]:
dec_fe = engineer_features(dec_clean, FIT_STATS)
print(dec_fe.shape)
dec_fe.head()  # quick visual check only, not required

In [ ]:
X_dec = dec_fe[feature_cols_b].copy()
for col in cat_features_b:
    X_dec[col] = X_dec[col].astype("category")

dec_predictions = lgb_model_b.predict(X_dec, num_iteration=lgb_model_b.best_iteration)

print(pd.Series(dec_predictions).describe())  # sanity check: no negatives, reasonable range, 31 values

count     31.000000
mean     842.913086
std       14.411524
min      800.124370
25%      841.568848
50%      849.500524
75%      852.020926
max      854.386102
dtype: float64


In [ ]:
dec_out = dec_raw.copy()
dec_out["predicted_rate"] = dec_predictions

# sanity checks before saving
assert list(dec_out.columns) == ["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]
assert len(dec_out) == 31
assert (dec_out["predicted_rate"] > 0).all()

os.makedirs("outputs", exist_ok=True)
# Note: the assessment README describes filling predicted_rate directly into
# december_chart_inputs.csv. This notebook instead writes the filled rows to a
# separate file with the same 7 columns, kept apart from the raw inputs.
# score.py accepts either path -- see repo README for details.
dec_out.to_csv("outputs/december_predictions.csv", index=False)
print(dec_out)

In [ ]:
# Final scorer run: validates both submission files and regenerates
# scorer_results/candidate_december.png (used in the report, Section 7)
!python score.py --predictions outputs/validation_predictions.csv --december-predictions outputs/december_predictions.csv

# Additional Validation & Analysis

Supporting analysis referenced in the report: R² comparison, a baseline ablation, prediction intervals, rolling-origin cross-validation, hyperparameter tuning, and an investigation into the December 28-30 dip. None of this is required for the two submission files above — it's the evidence behind the modeling decisions.

In [ ]:
from sklearn.metrics import r2_score

r2_a = r2_score(y_holdout, preds_lgb)
r2_b = r2_score(y_holdout, preds_lgb_b)
print(f"Head A R²: {r2_a:.4f}")
print(f"Head B R²: {r2_b:.4f}")

Head A R²: 0.8174
Head B R²: 0.8170


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, r2_score

def eval_preds(y_true, y_pred, label):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae  = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    r2   = r2_score(y_true, y_pred)
    print(f"{label:38s} RMSE={rmse:8.2f}  MAE={mae:8.2f}  MAPE={mape:6.2f}%  R2={r2:.4f}")
    return {"label": label, "rmse": rmse, "mae": mae, "mape": mape, "r2": r2}

def run_baselines(X_tr, X_te, y_tr, y_te, lgb_preds, lgb_label):
    results = []
    results.append(eval_preds(y_te, np.full(len(y_te), y_tr.mean()), "Baseline: global mean"))

    lr_dist = LinearRegression().fit(X_tr[["distance"]], y_tr)
    results.append(eval_preds(y_te, lr_dist.predict(X_te[["distance"]]), "Baseline: distance-only linear reg"))

    results.append(eval_preds(y_te, X_te["lane_mean_shrunk"], "Baseline: lane_mean_shrunk lookup (no ML)"))

    lr_feats = ["distance", "haversine_mi", "weight", "weight_to_distance", "lane_mean_shrunk"]
    lr_multi = LinearRegression().fit(X_tr[lr_feats], y_tr)
    results.append(eval_preds(y_te, lr_multi.predict(X_te[lr_feats]), "Baseline: multi-feature linear reg"))

    results.append(eval_preds(y_te, lgb_preds, lgb_label))
    return pd.DataFrame(results)

print("=== Head A baselines ===")
baseline_df_a = run_baselines(X_train, X_holdout, y_train, y_holdout, preds_lgb, "Head A: LightGBM (full features)")

print("\n=== Head B baselines ===")
baseline_df_b = run_baselines(X_train_b, X_holdout_b, y_train, y_holdout, preds_lgb_b, "Head B: LightGBM (no market signals)")

=== Head A baselines ===
Baseline: global mean                  RMSE= 1528.57  MAE= 1179.80  MAPE= 86.35%  R2=-0.0000
Baseline: distance-only linear reg     RMSE=  668.21  MAE=  192.77  MAPE= 10.25%  R2=0.8089
Baseline: lane_mean_shrunk lookup (no ML) RMSE=  946.67  MAE=  587.34  MAPE= 41.56%  R2=0.6164
Baseline: multi-feature linear reg     RMSE=  669.83  MAE=  200.44  MAPE= 10.71%  R2=0.8080
Head A: LightGBM (full features)       RMSE=  653.25  MAE=  120.61  MAPE=  5.20%  R2=0.8174

=== Head B baselines ===
Baseline: global mean                  RMSE= 1528.57  MAE= 1179.80  MAPE= 86.35%  R2=-0.0000
Baseline: distance-only linear reg     RMSE=  668.21  MAE=  192.77  MAPE= 10.25%  R2=0.8089
Baseline: lane_mean_shrunk lookup (no ML) RMSE=  946.67  MAE=  587.34  MAPE= 41.56%  R2=0.6164
Baseline: multi-feature linear reg     RMSE=  669.83  MAE=  200.44  MAPE= 10.71%  R2=0.8080
Head B: LightGBM (no market signals)   RMSE=  653.83  MAE=  124.19  MAPE=  5.50%  R2=0.8170


In [ ]:
# P10/P90 prediction intervals, for the report -- not part of the official submission
import lightgbm as lgb
import numpy as np
import pandas as pd

def train_quantile_model(X_train, y_train, X_val, y_val, cat_features, alpha, base_params):
    params = dict(base_params)
    params["objective"] = "quantile"
    params["alpha"] = alpha
    params["metric"] = "quantile"
    train_set = lgb.Dataset(X_train, y_train, categorical_feature=cat_features)
    val_set   = lgb.Dataset(X_val, y_val, categorical_feature=cat_features, reference=train_set)
    model = lgb.train(
        params, train_set, num_boost_round=4000,
        valid_sets=[val_set], valid_names=["holdout"],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    return model

base_q_params = {
    "learning_rate": 0.05, "num_leaves": 64, "max_depth": 8,
    "min_data_in_leaf": 30, "feature_fraction": 0.85,
    "bagging_fraction": 0.85, "bagging_freq": 5, "seed": 42, "verbose": -1,
}

# --- Head A: bands for validation.csv ---
q10_a = train_quantile_model(X_train_lgb, y_train, X_holdout_lgb, y_holdout, cat_features, 0.10, base_q_params)
q90_a = train_quantile_model(X_train_lgb, y_train, X_holdout_lgb, y_holdout, cat_features, 0.90, base_q_params)

holdout_p10_a = q10_a.predict(X_holdout_lgb, num_iteration=q10_a.best_iteration)
holdout_p90_a = q90_a.predict(X_holdout_lgb, num_iteration=q90_a.best_iteration)
coverage_a = ((y_holdout.values >= holdout_p10_a) & (y_holdout.values <= holdout_p90_a)).mean()
print(f"Head A - empirical P10-P90 coverage on holdout: {coverage_a*100:.1f}% (target ~80%)")

val_p10 = q10_a.predict(X_val, num_iteration=q10_a.best_iteration)
val_p90 = q90_a.predict(X_val, num_iteration=q90_a.best_iteration)
result_with_bands = result.copy()
result_with_bands["predicted_rate_p10"] = val_p10
result_with_bands["predicted_rate_p90"] = val_p90
result_with_bands.to_csv("outputs/validation_predictions_with_bands.csv", index=False)  # for report only, NOT the official submission
print(result_with_bands.head())

# --- Head B: bands for December ---
q10_b = train_quantile_model(X_train_b, y_train, X_holdout_b, y_holdout, cat_features_b, 0.10, base_q_params)
q90_b = train_quantile_model(X_train_b, y_train, X_holdout_b, y_holdout, cat_features_b, 0.90, base_q_params)

holdout_p10_b = q10_b.predict(X_holdout_b, num_iteration=q10_b.best_iteration)
holdout_p90_b = q90_b.predict(X_holdout_b, num_iteration=q90_b.best_iteration)
coverage_b = ((y_holdout.values >= holdout_p10_b) & (y_holdout.values <= holdout_p90_b)).mean()
print(f"Head B - empirical P10-P90 coverage on holdout: {coverage_b*100:.1f}% (target ~80%)")

dec_p10 = q10_b.predict(X_dec, num_iteration=q10_b.best_iteration)
dec_p90 = q90_b.predict(X_dec, num_iteration=q90_b.best_iteration)
dec_out_with_bands = dec_out.copy()
dec_out_with_bands["predicted_rate_p10"] = dec_p10
dec_out_with_bands["predicted_rate_p90"] = dec_p90
dec_out_with_bands.to_csv("outputs/december_predictions_with_bands.csv", index=False)  # for report only, NOT what you feed score.py
print(dec_out_with_bands)

In [ ]:
# Rolling-origin (walk-forward) cross-validation across three independent monthly
# holdouts, to confirm the single Oct-2025 holdout above wasn't a lucky split
# (report Section 2.1)
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error
import pandas as pd

rolling_windows = [pd.Timestamp(2025, 8, 1), pd.Timestamp(2025, 9, 1), pd.Timestamp(2025, 10, 1)]
cv_results = []

for holdout_start_cv in rolling_windows:
    holdout_end_cv = holdout_start_cv + pd.offsets.MonthBegin(1)
    tr = df_clean[df_clean["date"] < holdout_start_cv].copy()
    ho = df_clean[(df_clean["date"] >= holdout_start_cv) & (df_clean["date"] < holdout_end_cv)].copy()
    if len(tr) == 0 or len(ho) == 0:
        continue

    stats_cv = fit_feature_stats(tr)  # fit ONLY on this fold's train — stays leak-safe
    tr_fe = engineer_features(tr, stats_cv)
    ho_fe = engineer_features(ho, stats_cv)

    feat_cols_cv = [c for c in tr_fe.columns if c not in ["load_id", "posted_rate", "date", "rate_per_mile"]]
    cat_cols_cv  = [c for c in cat_features if c in feat_cols_cv]

    X_tr, y_tr = tr_fe[feat_cols_cv].copy(), tr_fe["posted_rate"]
    X_ho, y_ho = ho_fe[feat_cols_cv].copy(), ho_fe["posted_rate"]
    for c in cat_cols_cv:
        X_tr[c] = X_tr[c].astype("category")
        X_ho[c] = X_ho[c].astype("category")

    train_set = lgb.Dataset(X_tr, y_tr, categorical_feature=cat_cols_cv)
    val_set   = lgb.Dataset(X_ho, y_ho, categorical_feature=cat_cols_cv, reference=train_set)
    model_cv = lgb.train(
        lgb_params, train_set, num_boost_round=8000,
        valid_sets=[val_set], valid_names=["holdout"],
        callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)],
    )
    preds_cv = model_cv.predict(X_ho, num_iteration=model_cv.best_iteration)
    mape_cv = mean_absolute_percentage_error(y_ho, preds_cv) * 100
    cv_results.append({"holdout_month": holdout_start_cv.strftime("%Y-%m"), "mape": mape_cv, "n_holdout": len(ho)})
    print(f"Holdout {holdout_start_cv.strftime('%Y-%m')}: MAPE = {mape_cv:.2f}%  (n={len(ho)})")

cv_df = pd.DataFrame(cv_results)
print(cv_df)
print(f"\nMean MAPE across {len(cv_df)} rolling windows: {cv_df['mape'].mean():.2f}% +/- {cv_df['mape'].std():.2f}%")

## Hyperparameter Tuning with Optuna

The manually-tuned `lgb_params` above (Section "Model Selection") were checked against an automated search, to see whether they were leaving accuracy on the table.

- **Search space:** `learning_rate`, `num_leaves`, `max_depth`, `min_data_in_leaf`, `feature_fraction`, `bagging_fraction`, `bagging_freq` — the same knobs documented inline in the Head A training cell above
- **Objective:** minimize MAPE on the October 2025 holdout (same metric, same split used everywhere else in this notebook)
- **Budget:** 40 trials, TPE sampler (Optuna's default Bayesian-style sampler), seeded for reproducibility

**Result:** best trial found MAPE = 4.87%, a ~0.3 percentage-point improvement over the manually-tuned 5.20%. This confirms the manual params were already in a reasonable region of the search space rather than being badly mistuned — the gain is real but modest, so the manually-tuned params (not the Optuna-best ones) are what's actually used for the submitted `validation_predictions.csv` and `december_predictions.csv` above, to keep the two heads directly comparable on identical hyperparameters as documented in the report.

In [ ]:
# Optuna hyperparameter search against the manually-tuned Head A params above
!pip install optuna -q
import optuna
import lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error

def objective(trial):
    params = {
        "objective": "mape", "metric": "mape",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 128),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 100),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "seed": 42, "verbose": -1,
    }
    train_set = lgb.Dataset(X_train_lgb, y_train, categorical_feature=cat_features)
    val_set   = lgb.Dataset(X_holdout_lgb, y_holdout, categorical_feature=cat_features, reference=train_set)
    model = lgb.train(
        params, train_set, num_boost_round=4000,
        valid_sets=[val_set], valid_names=["holdout"],
        callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
    )
    preds = model.predict(X_holdout_lgb, num_iteration=model.best_iteration)
    return mean_absolute_percentage_error(y_holdout, preds) * 100

study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Current manual-params holdout MAPE:", f"{mape_lgb:.2f}%")
print("Best Optuna holdout MAPE:          ", f"{study.best_value:.2f}%")
print("Best params found:", study.best_params)
print(f"\nImprovement: {mape_lgb - study.best_value:.2f}pp")

## December extrapolation check

December 2025 falls entirely outside the training date range (Jan-Sep). The cell below checks whether Head B's raw `week_of_year`/`month` features are being extrapolated in a way that could explain the December 28-30 dip in the forecast, by comparing against a cyclical (sin/cos) encoding of the same calendar features.

**Result:** the cyclical encoding changes the December predictions substantially (mean drops from ~843 to ~790) without improving holdout MAPE (5.54% vs. 5.50%), so it was not adopted — `outputs/december_predictions.csv` above uses the original (non-cyclical) Head B model. This dip is flagged as an open item in the report (Section 7.1) rather than resolved here.

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import mean_absolute_percentage_error

print("Training week_of_year range:", df_train_fe["week_of_year"].min(), "-", df_train_fe["week_of_year"].max())
print("December week_of_year values:", sorted(dec_fe["week_of_year"].unique()))
print("Training month values seen:", sorted(df_train_fe["month"].unique()))
print("December month value:", dec_fe["month"].unique())
print("\nCurrent Head B December predictions:")
print(pd.Series(dec_predictions).describe())

def add_cyclical_calendar(df):
    df = df.copy()
    df["week_sin"]  = np.sin(2 * np.pi * df["week_of_year"] / 52)
    df["week_cos"]  = np.cos(2 * np.pi * df["week_of_year"] / 52)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    return df

df_train_fe_cyc   = add_cyclical_calendar(df_train_fe)
df_holdout_fe_cyc = add_cyclical_calendar(df_holdout_fe)
dec_fe_cyc        = add_cyclical_calendar(dec_fe)

feature_cols_b_cyc = [c for c in feature_cols_b if c not in ("week_of_year", "month")] + \
                      ["week_sin", "week_cos", "month_sin", "month_cos"]
cat_features_b_cyc = [c for c in cat_features_b if c in feature_cols_b_cyc]

X_train_b_cyc = df_train_fe_cyc[feature_cols_b_cyc].copy()
X_holdout_b_cyc = df_holdout_fe_cyc[feature_cols_b_cyc].copy()
for c in cat_features_b_cyc:
    X_train_b_cyc[c] = X_train_b_cyc[c].astype("category")
    X_holdout_b_cyc[c] = X_holdout_b_cyc[c].astype("category")

train_set_cyc = lgb.Dataset(X_train_b_cyc, y_train, categorical_feature=cat_features_b_cyc)
val_set_cyc   = lgb.Dataset(X_holdout_b_cyc, y_holdout, categorical_feature=cat_features_b_cyc, reference=train_set_cyc)

lgb_model_b_cyc = lgb.train(
    lgb_params_b, train_set_cyc, num_boost_round=8000,
    valid_sets=[val_set_cyc], valid_names=["holdout"],
    callbacks=[lgb.early_stopping(150), lgb.log_evaluation(200)],
)

preds_b_cyc = lgb_model_b_cyc.predict(X_holdout_b_cyc, num_iteration=lgb_model_b_cyc.best_iteration)
mape_b_cyc = mean_absolute_percentage_error(y_holdout, preds_b_cyc) * 100
print(f"\nHead B (cyclical calendar) holdout MAPE: {mape_b_cyc:.2f}%  (original Head B: {mape_b:.2f}%)")

X_dec_cyc = dec_fe_cyc[feature_cols_b_cyc].copy()
for c in cat_features_b_cyc:
    X_dec_cyc[c] = X_dec_cyc[c].astype("category")

dec_predictions_cyc = lgb_model_b_cyc.predict(X_dec_cyc, num_iteration=lgb_model_b_cyc.best_iteration)
print("\nDecember predictions - BEFORE (raw week_of_year/month):")
print(pd.Series(dec_predictions).describe())
print("\nDecember predictions - AFTER (cyclical calendar features):")
print(pd.Series(dec_predictions_cyc).describe())

Training week_of_year range: 1 - 40
December week_of_year values: [np.int64(1), np.int64(49), np.int64(50), np.int64(51), np.int64(52)]
Training month values seen: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9)]
December month value: [12]

Current Head B December predictions:
count     31.000000
mean     842.913086
std       14.411524
min      800.124370
25%      841.568848
50%      849.500524
75%      852.020926
max      854.386102
dtype: float64
Training until validation scores don't improve for 150 rounds
[200]	holdout's mape: 0.0600891
[400]	holdout's mape: 0.0574687
[600]	holdout's mape: 0.0568473
[800]	holdout's mape: 0.0565401
[1000]	holdout's mape: 0.056166
[1200]	holdout's mape: 0.0560527
[1400]	holdout's mape: 0.0558897
[1600]	holdout's mape: 0.0558369
[1800]	holdout's mape: 0.0557641
[2000]	holdout's mape: 0.0556894
[2200]	holdout's mape: 0.0556336
[2400]	holdout's mape: 0.0555803
[2600]	holdout's mape: 0.